In [ ]:
!nvidia-smi

In [ ]:
!pip install -U torchao diffusers transformers accelerate

In [ ]:
!pip install -q diffusers transformers accelerate

from diffusers import StableDiffusionPipeline
import torch, os
from PIL import Image

# Загружаем модель
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
).to("cuda")

# Промпты (разные темы, чтобы получить разнообразные текстуры)
prompts = [
    "a high quality photo of a cat", "a realistic photo of a coffee cup",
    "a beautiful mountain landscape", "a vintage car on a street",
    "a plate of food on a table", "a close-up of a flower",
    "a modern building facade", "a wooden texture background",
    "a portrait of a person", "a bowl of fruit",
] * 20  # 200 изображений

os.makedirs("/content/ai_raw", exist_ok=True)

for i, p in enumerate(prompts[:200]):
    img = pipe(p, num_inference_steps=25).images[0]
    img.save(f"/content/ai_raw/sd_{i:04d}.png")
    if (i+1) % 50 == 0:
        print(f"Сгенерировано {i+1}/200")

# Упаковываем в zip
!zip -r /content/ai_raw.zip /content/ai_raw

# Скачиваем архив на компьютер
from google.colab import files
files.download('/content/ai_raw.zip')
print("Готово! Архив скачивается.")

In [ ]:
!pip install -q diffusers transformers accelerate

from diffusers import StableDiffusionXLPipeline, DDPMPipeline
import torch, os
from PIL import Image
from tqdm import tqdm

os.makedirs("/content/ai_new", exist_ok=True)

# Промпты (возьмем разные)
prompts = [
    "a high quality photo of a cat", "a realistic photo of a coffee cup",
    "a beautiful mountain landscape", "a vintage car on a street",
    "a plate of food on a table", "a close-up of a flower",
    "a modern building facade", "a wooden texture background",
    "a portrait of a person", "a bowl of fruit",
] * 10  # 100 картинок на каждую модель

# 1. SDXL (Современная диффузия)
print("Генерация SDXL...")
pipe_xl = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True
).to("cuda")

for i, p in enumerate(prompts[:100]):
    img = pipe_xl(p, num_inference_steps=20).images[0]
    # Сразу нарезаем на куски 256x256
    w, h = img.size
    TILE_SIZE, OVERLAP = 256, 64
    step = TILE_SIZE - OVERLAP
    for y in range(0, h - TILE_SIZE + 1, step):
        for x in range(0, w - TILE_SIZE + 1, step):
            tile = img.crop((x, y, x + TILE_SIZE, y + TILE_SIZE))
            tile.save(f"/content/ai_new/sdxl_{i:04d}_{x}_{y}.png")
    if (i+1) % 20 == 0: print(f"SDXL: {i+1}/100")

#2. DDPM (Классическая диффузия)
print("Генерация DDPM...")
pipe_ddpm = DDPMPipeline.from_pretrained("google/ddpm-celebahq-256").to("cuda")
for i in range(100):
    img = pipe_ddpm().images[0]
    w, h = img.size
    TILE_SIZE, OVERLAP = 256, 64
    step = TILE_SIZE - OVERLAP
    for y in range(0, h - TILE_SIZE + 1, step):
        for x in range(0, w - TILE_SIZE + 1, step):
            tile = img.crop((x, y, x + TILE_SIZE, y + TILE_SIZE))
            tile.save(f"/content/ai_new/ddpm_{i:04d}_{x}_{y}.png")
    if (i+1) % 20 == 0: print(f"DDPM: {i+1}/100")

!zip -r /content/ai_new_tiles.zip /content/ai_new
from google.colab import files
files.download('/content/ai_new_tiles.zip')
print("Готово! Скачайте архив и распакуйте в папку AI_Research/ai/tiles_256")

In [ ]:
# 1. Клонируем официальный репозиторий NVIDIA
!git clone -q https://github.com/NVlabs/stylegan2-ada-pytorch.git
import sys
sys.path.append('/content/stylegan2-ada-pytorch')
import torch, dnnlib, legacy
from PIL import Image
import os
from tqdm import tqdm

# 2. Скачиваем предобученные веса (лица FFHQ 512x512)
!wget -q https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl -O /content/ffhq.pkl

# 3. Загружаем модель
device = torch.device('cuda')
with dnnlib.util.open_url('/content/ffhq.pkl') as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device)

# 4. Генерируем и нарезаем
os.makedirs("/content/gan", exist_ok=True)
TILE_SIZE, OVERLAP = 256, 64
step = TILE_SIZE - OVERLAP
count = 0

print("Генерация GAN (StyleGAN2-ADA)...")
for i in tqdm(range(100)): # 100 картинок
    z = torch.randn([1, G.z_dim]).to(device)
    img_tensor = G(z, None) # Генерируем 512x512
    img_tensor = (img_tensor.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    img = Image.fromarray(img_tensor[0].cpu().numpy())

    w, h = img.size
    for y in range(0, h - TILE_SIZE + 1, step):
        for x in range(0, w - TILE_SIZE + 1, step):
            tile = img.crop((x, y, x + TILE_SIZE, y + TILE_SIZE))
            tile.save(f"/content/gan/gan_{count:06d}.png")
            count += 1
print(f"GAN: нарезано {count} фрагментов 256x256.")

# 5. Упаковка
!zip -r /content/gan_tiles_final.zip /content/gan
from google.colab import files
files.download('/content/gan_tiles_final.zip')
print("Архив с GAN скачивается!")

In [ ]:
!pip install -q diffusers transformers accelerate

import torch
from diffusers import AutoencoderKL
from PIL import Image
import os
from tqdm import tqdm

# Загружаем VAE
print("Загрузка VAE (stabilityai/sd-vae-ft-mse)...")
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float16
).cuda()
vae.eval()

os.makedirs("/content/vae", exist_ok=True)

TILE_SIZE, OVERLAP = 256, 64
step = TILE_SIZE - OVERLAP
count = 0

print("Генерация VAE...")
with torch.no_grad():
    for i in tqdm(range(100)):  # 100 картинок
        # Генерируем случайный шум в латентном пространстве (4, 64, 64) — для 512×512
        latent = torch.randn(1, 4, 64, 64).cuda().half()

        # Декодируем в изображение
        decoded = vae.decode(latent).sample

        # Переводим в [0, 1] и PIL
        decoded = (decoded / 2 + 0.5).clamp(0, 1)
        img_tensor = decoded.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img = Image.fromarray((img_tensor * 255).astype('uint8'))

        # Нарезаем на фрагменты 256×256
        w, h = img.size
        for y in range(0, h - TILE_SIZE + 1, step):
            for x in range(0, w - TILE_SIZE + 1, step):
                tile = img.crop((x, y, x + TILE_SIZE, y + TILE_SIZE))
                tile.save(f"/content/vae/vae_{count:06d}.png")
                count += 1

        if (i+1) % 20 == 0:
            print(f"  Сгенерировано {i+1}/100, нарезано {count} фрагментов")

print(f"\nГотово! VAE: нарезано {count} фрагментов 256×256.")

# Упаковка
!zip -r /content/vae_tiles_final.zip /content/vae
from google.colab import files
files.download('/content/vae_tiles_final.zip')
print("Архив с VAE скачивается!")

In [ ]:
!pip install -q diffusers transformers accelerate

import torch
from diffusers import AutoencoderKL
from PIL import Image
import os
import zipfile
from tqdm import tqdm
from google.colab import files
from torchvision import transforms

# 1. ЗАГРУЗКА АРХИВА С РЕАЛЬНЫМИ ФОТО
print("Загрузите zip-архив с вашими реальными фото (tiles_256.zip):")
uploaded = files.upload()

for fname in uploaded.keys():
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/real_photos')
print("Архив распакован.")

# 2. ЗАГРУЗКА МОДЕЛИ VAE
print("Загрузка VAE...")
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
    torch_dtype=torch.float16
).cuda()
vae.eval()

os.makedirs('/content/vae', exist_ok=True)
TILE_SIZE, OVERLAP = 256, 64
step = TILE_SIZE - OVERLAP

# 3. ГЕНЕРАЦИЯ ИЗ ШУМА (100 картинок)
print("\n--- Генерация VAE из шума ---")
count_noise = 0
with torch.no_grad():
    for i in tqdm(range(100), desc="Шум"):
        latent = torch.randn(1, 4, 64, 64).cuda().half()
        decoded = vae.decode(latent).sample
        decoded = (decoded / 2 + 0.5).clamp(0, 1)
        img_tensor = decoded.squeeze(0).permute(1, 2, 0).cpu().numpy()
        img = Image.fromarray((img_tensor * 255).astype('uint8'))

        w, h = img.size
        for y in range(0, h - TILE_SIZE + 1, step):
            for x in range(0, w - TILE_SIZE + 1, step):
                tile = img.crop((x, y, x + TILE_SIZE, y + TILE_SIZE))
                tile.save(f"/content/vae/vae_noise_{count_noise:06d}.png")
                count_noise += 1
print(f"Из шума: {count_noise} фрагментов.")

# 4. РЕКОНСТРУКЦИЯ РЕАЛЬНЫХ ФОТО
print("\n--- Реконструкция реальных фото ---")

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# Ищем все реальные фото в распакованной папке
all_real_files = []
for root, dirs, files_ in os.walk('/content/real_photos'):
    for f in files_:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            all_real_files.append(os.path.join(root, f))

print(f"Найдено реальных фото: {len(all_real_files)}")

count_recon = 0
with torch.no_grad():
    for path in tqdm(all_real_files[:500], desc="Реконструкция"):
        try:
            img = Image.open(path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).cuda().half()

            # Кодируем и декодируем
            posterior = vae.encode(img_tensor).latent_dist
            z = posterior.sample()
            recon = vae.decode(z).sample
            recon = (recon / 2 + 0.5).clamp(0, 1)

            recon_img = recon.squeeze(0).permute(1, 2, 0).cpu().numpy()
            recon_img = (recon_img * 255).astype('uint8')
            Image.fromarray(recon_img).save(f'/content/vae/vae_recon_{count_recon:06d}.png')
            count_recon += 1
        except Exception as e:
            print(f"Ошибка с {path}: {e}")

print(f"Реконструировано: {count_recon} изображений.")

# 5. УПАКОВКА И СКАЧИВАНИЕ
total = count_noise + count_recon
print(f"\n=== ИТОГО VAE-фрагментов: {total} ===")

!zip -r /content/vae_final.zip /content/vae
files.download('/content/vae_final.zip')
print("Архив vae_final.zip скачивается!")

In [ ]:
!pip install -q diffusers transformers accelerate

from diffusers import DDPMPipeline
import torch, os
from tqdm import tqdm

os.makedirs("/content/ddpm_1000", exist_ok=True)

# DDPM (Генерируем 1000 картинок)
print("Генерация DDPM (1000 штук). Это займет около 20-30 минут на T4...")
pipe_ddpm = DDPMPipeline.from_pretrained("google/ddpm-celebahq-256").to("cuda")

# Начинаем нумерацию с 100, чтобы не перезаписать старые
for i in tqdm(range(1000), desc="DDPM generation"):
    img = pipe_ddpm().images[0]
    w, h = img.size
    TILE_SIZE, OVERLAP = 256, 64
    step = TILE_SIZE - OVERLAP

    # Нарезаем на куски (для картинки 256x256 получится ровно 1 фрагмент)
    for y in range(0, h - TILE_SIZE + 1, step):
        for x in range(0, w - TILE_SIZE + 1, step):
            tile = img.crop((x, y, x + TILE_SIZE, y + TILE_SIZE))
            # Сохраняем: нумерация идет с 100 до 1099
            tile.save(f"/content/ddpm_1000/ddpm_{i+100:04d}_{x}_{y}.png")

# Архивация и скачивание
!zip -q -r /content/ddpm_1000_tiles.zip /content/ddpm_1000
from google.colab import files
files.download('/content/ddpm_1000_tiles.zip')
print("Готово! Архив скачан. Распакуйте его в папку AI_Research/ai/tiles_256/ddpm/")

In [ ]:
!zip -q -r /content/ddpm_24_tiles.zip /content/ddpm_1000
from google.colab import files
files.download('/content/ddpm_24_tiles.zip')
print("Готово! Архив скачан.")